# CCI V2.1-D2 — Transformer Challenge (Kaggle execution)

Step 4 of the seven-step cycle authorized by ADR-010, `challenge_best_classical_with_compact_transformer`: a compact transformer challenge to the D1 classical winner `word_char_tfidf_union_40000_60000_c_1_hard_negative`, whose outer critical F1 was **0.386899**.

**Controlled design (ADR-012):** everything that is not the stage-A model family is held identical to the D1 winner — the same `inner_fit` scope, the same hard-negative training pool (`generate_hard_negative_indices` with the same parameters), the same inner calibration window and exact threshold search (`search_detector_threshold_exact`), the same outer evaluation window, and the same hierarchical critical-override architecture. Only the stage-A model family varies: `distilbert-base-uncased` in place of the TF-IDF word+char union with LinearSVC.

**Pre-registered decision rule:** three seeds, 42, 43, and 44, are run as replicates of one model, never as distinct models. The **median** of the three outer critical F1 scores is reported, together with the full metric vector of the seed attaining that median; reporting the best of the three seeds is forbidden. The transformer displaces the classical incumbent only if the reported seed's outer evaluation clears critical F1 **0.425599** — a **+0.0387** increment over the incumbent, equal to two bootstrap standard deviations — while also holding critical precision at or above **0.434286**. Any other outcome is `CLASSICAL_WINNER_STANDS`, a legitimate, pre-accepted result and not a failure.

**Boundary:** the bundle carries code, frozen configs, and the frozen S7 fallback package. The only data file is the development-only `scientific.parquet` (`train` + `validation`). `test`, `stress`, and `monitor` remain sealed and have no unlock path in this code.

All computational logic lives in the shipped package (`consumer_complaint_intelligence.kaggle_execution` and `consumer_complaint_intelligence.v2_transformer`); cells only orchestrate and print aggregate evidence.

In [ ]:
%pip install --quiet scikit-learn==1.9.0 imbalanced-learn==0.14.2
try:
    import transformers  # noqa: F401
except ImportError:
    %pip install --quiet transformers


In [ ]:
import sys
import zipfile
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/temp/project")
OUTPUT_ROOT = Path("/kaggle/working")


def _input_listing(limit=200):
    return [str(path) for path in sorted(INPUT_ROOT.rglob("*"))[:limit]]


manifests = [
    path
    for path in sorted(INPUT_ROOT.rglob("kaggle_bundle_manifest.json"))
    if (path.parent / "src").is_dir()
]
if manifests:
    bundle_root = manifests[0].parent
else:
    zips = sorted(INPUT_ROOT.rglob("cci-v2-bundle.zip"))
    if not zips:
        raise FileNotFoundError(
            f"No bundle manifest or zip under {INPUT_ROOT}; "
            f"mounted: {_input_listing()}"
        )
    bundle_root = Path("/kaggle/temp/bundle_extracted")
    bundle_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zips[0]) as archive:
        archive.extractall(bundle_root)

caches = sorted(INPUT_ROOT.rglob("scientific.parquet"))
if not caches:
    raise FileNotFoundError(
        f"No scientific.parquet under {INPUT_ROOT}; "
        f"mounted: {_input_listing()}"
    )
CACHE_FILE = caches[0]
print("bundle_root:", bundle_root)
print("cache_file:", CACHE_FILE)

sys.path.insert(0, str(bundle_root / "src"))
from consumer_complaint_intelligence import kaggle_execution as kx

kx.assert_pinned_environment()
print(kx.report_gpu())
# Prove the accelerator can run a kernel before spending ~9 minutes
# building the hard-negative pool. A torch build without kernels for
# this device reports cuda_available True and dies on first launch.
print(kx.assert_usable_gpu())


In [ ]:
# Warm the model cache before staging so a download failure surfaces in
# about a minute instead of after the benchmark has already started.
import json as _json

_cfg = _json.loads(
    (bundle_root / "config" / "v2_d2_execution.json").read_text(
        encoding="utf-8"
    )
)
MODEL_ID = _cfg["model"]["model_id"]
from transformers import AutoModelForSequenceClassification, AutoTokenizer

_tok = AutoTokenizer.from_pretrained(MODEL_ID)
_mdl = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=2
)
print(
    {
        "model_id": MODEL_ID,
        "vocab_size": _tok.vocab_size,
        "parameters": sum(p.numel() for p in _mdl.parameters()),
    }
)
del _mdl


In [ ]:
staging = kx.stage_project(bundle_root, CACHE_FILE, WORK_ROOT)
print(staging)
print(kx.preflight_d2(WORK_ROOT))

In [ ]:
result = kx.run_full_d2(WORK_ROOT)
reported = result.get("reported") or {}
outer = reported.get("outer") or {}
metrics = outer.get("metrics") or outer
decision = result.get("decision") or {}
print(
    {
        "status": result.get("status"),
        "complete": result.get("complete"),
        "runtime_seconds": result.get("runtime_seconds"),
        "reported_seed": reported.get("seed"),
        "outer_critical_f1": metrics.get("critical_f1"),
        "outer_critical_precision": metrics.get("critical_precision"),
        "critical_f1_vs_incumbent": reported.get(
            "critical_f1_vs_incumbent"
        ),
        "critical_f1_vs_fallback": reported.get(
            "critical_f1_vs_fallback"
        ),
        "seed_spread": result.get("seed_spread"),
        "decision_outcome": decision.get("outcome"),
        "blocked_reason": decision.get("blocked_reason"),
    }
)


In [ ]:
print(kx.collect_outputs_d2(WORK_ROOT, OUTPUT_ROOT))

## Retrieval and outcome

The staged tree, and any fitted transformer weights, live under `/kaggle/temp` and are discarded with the session. Only the two aggregate JSON files are persisted as notebook output:

- `v2_transformer_challenge.json` → local `temp/v2/`
- `v2_transformer_results.json` → local `config/`

The published artifact is aggregate-only: no narratives, identifiers, individual margins, or fitted weights are persisted. A `CLASSICAL_WINNER_STANDS` outcome is a legitimate, pre-registered, published result, not a failure of this run — the D2 question is answered either way. No sealed partition (`test`, `stress`, or `monitor`) is touched by this notebook or by the package it calls.

Download the outputs (`kaggle kernels output`) and validate locally with `validate_d2_manifest` plus the `tests/test_v2_*` suite before any package-freeze decision.